In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

# Data filtering

In [ ]:
# Helper functions

import re
def has_answer_marker(record):
    response = str(record['response_vi'])
    return 'Đáp án là' in response or 'Câu trả lời là' in response


# Since boxed data has many complex answers
def extract_boxed(text):
    match = re.search(r'\\boxed\{([^}]+)\}', text)
    return match.group(1) if match else None

def is_numeric(value):
    if value is None:
        return False
    try:
        float(value.replace(',', ''))
        return True
    except ValueError:
        return False
def is_salvageable(record, boxed_value):
    if boxed_value is None:  # no boxed, keep for now
        return True
    return is_numeric(boxed_value)
# strip the ollar sign
def strip_dollar(text):
    return text.replace('$', '')
# convert the cdots
def clean_cdot(text):
    return text.replace('\\cdot', '*')

import re




# All plain-text substitutions for clean_latex
LATEX_REPLACEMENTS = [
    ('\\times',   '*'),
    ('\\div',     '/'),
    ('\\circ',    ' độ'),
    ('\\leq',     '<='),
    ('\\le',      '<='),
    ('\\geq',     '>='),
    ('\\ge',      '>='),
    ('\\dots',    '...'),
    ('\\ldots',   '...'),
    ('\\pm',      'cộng hoặc trừ'),
    ('\\dfrac',   '\\frac'),
    ('\\tfrac',   '\\frac'),
    ('\\cfrac',   '\\frac'),
    ('\\approx',  '≈'),
    ('\\neq',     '!='),
    ('\\ne',      '!='),
    ('\\angle',   'góc'),
    ('\\triangle','tam giác'),
    ('\\tam',     'tam'),
    ('\\time',    '*'),
    ('\\ngụ ý',   'ngụ ý'),
    ('\\quad',    ' '),
    ('\\ast',     '*'),
    ('\\square',  ''),
    ('\\deg',     ' độ'),
    ('\\min',     'min'),
    ('\\sim',     '~'),
    ('\\S',       ''),
    # clean remaining
    ('\\sec',      'sec'),
    ('\\dagger',   ''),
    ('\\cong',     '≅'),
    ('\\parallel', '//'),
    ('\\Gamma',    'Γ'),
    ('\\sigma',    'σ'),
    ('\\perp',     '⊥'),
    ('\\lvert',    '|'),
    ('\\rvert',    '|'),
    ('\\max', 'max'),

]

def clean_latex(text):
    for pattern, replacement in LATEX_REPLACEMENTS:
        text = text.replace(pattern, replacement)
    return text


UNSOLVABLE_PHRASES = [
    'không thể xác định',
    'không được đưa ra',
    'không đủ thông tin',
    'không có đủ dữ liệu',
    'không thể xảy ra',  # add this
]

def is_unsolvable(record):
    response = str(record['response_vi'])
    return any(phrase in response for phrase in UNSOLVABLE_PHRASES)


def fix_boxed(text):
    text = text.replace('=ed{', '\\boxed{')
    text = text.replace('\\đóng hộp{', '\\boxed{')
    text = text.replace('\\Box{', '\\boxed{')
    text = text.replace('\\Box', '\\boxed')
    # only replace \box if not already \boxed
    text = re.sub(r'\\box(?!ed)', '\\\\boxed', text)
    return text
def convert_frac(text, max_iter=10):
    pattern = r'\\frac\s*\{([^{}]*)\}\s*\{([^{}]*)\}'
    for _ in range(max_iter):
        new = re.sub(pattern, r'(\1)/(\2)', text)
        if new == text:
            break
        text = new
    return text

# All patterns that mark a problem as too complex to keep
COMPLEX_PATTERNS = [
    '[asy]',
    '\\begin{align',
    '\\log', '\\sqrt', '\\binom',
    '\\sum', '\\prod', '\\lim', '\\int', '\\infty',
    '\\text{', 'H_{', '\\begin{pmatrix}',
    '\\pmod', '\\equiv',
    '\\sin', '\\cos', '\\tan', '\\pi',
    '\\alpha', '\\beta', '\\theta', '\\nabla', '\\mathbf',
    '\\begin', '\\multicolumn', '\\cline', '\\overline',
    '\\lfloor', '\\rfloor',
    '\\dbinom', '\\choose', '\\omega', '\\gcd', '\\mod',
    '\\cap', '\\cup', '\\displaystyle', '\\Rightarrow',
    '\\Diamond', '\\spadesuit', '\\star',
    '\\left', '<=ft', '\\right',
    '\\clubsuit', '\\diamondsuit', '\\odot',
    '\\lceil', '\\rceil', '\\overrightarrow', '\\operatorname',
    '\\Delta', '\\Theta', '\\implies', '\\oplus',
    '\\mathrm', '\\textrm', '\\rm', '\\heartsuit',
    '\\arcsin', '\\phi', '\\delta', '\\partial',
    '\\stackrel', '\\cancel', '\\frown', '\\bold',
    '\\underline', '\\mbox', '\\mathcal', '\\bmod',
    '\\det', '\\mathbb', '\\textnormal', '\\color',
    '\\in','\\text','\\cotang',
    '\\widehat',
    '\\vdots',
    '\\mapsto',
    '\\]',   # catches the \[ ... \] display math blocks
    '\\[',
    'x_1', 'x_2', 'y_1', 'y_2',  # coordinate geometry formulas
    'A(x', 'B(x', 'C(x',          # partial fractions
    'x + 1)(x + 2)',               # polynomial expansion
    '^{',   # complex exponents like ^{\frac35}, ^{10}, ^{mn}
    # Series / sequences
    'S = n/2',
    'a_1',
    'hiệu chung',
    'dãy số học',
    'công thức S',
    
    # Polynomials / algebra
    'x^2', 'x^3', 'x^4', 'x^5',
    ')(x',              # factored form (x+3)(x-1)
    'x^2+', 'x^2-',
    
    # Systems of equations
    'gọi s ', 'gọi l ',
    'hai phương trình',
    'thuật toán Euclide',
    'ước chung lớn nhất',
    
    # Variables being solved for
    'Gọi x ', 'gọi x ',   # "let x ="
    'Gọi y ', 'gọi y ',
    'Gọi n ', 'gọi n ',
    
    # Step-by-step algorithms
    'Bước 1:',
    'Bước 2:',
    'cực tiểu hóa',   # "minimize" → optimization problem
    'tỉ lệ thuận',    # "proportional to" → ratio/algebra problem  
    'giai đoạn',      # "stage/phase" → sequence pattern problem
    'trung bình X',   # literal unknown in word problem
    'V = lwh',        # volume formula
    '= lwh',          # catches any volume variant
    'chữ số đầu tiên', # "first digit" → counting/combinatorics problem
    'lựa chọn',       # "choices" → combinatorics
    # 'tốc độ',         # "speed" → distance/rate problems (d = v*t, too complex)
    # 'dặm',            # "miles" → rate problems
    # 'km/h', 'mph',
    # low count — safe to drop
    'ma trận',      # matrix
    'mod ',          # modular
    'S = n/2',       # series
    'a_1',           # series
    'dãy số học',    # series
    'Bước 1:',       # many steps
    # many_steps
    '4 mỗi lần',        # "4 at a time" → step/remainder problem
    
    # series_sequence  
    '3^0', '3^1', '3^2',   # geometric series
    'S_n = n(n+1)/2',      # sum formula
    'n(n+1)/2',            # same formula variant
    
    # matrix/divisibility
    'chia hết cho 31',     # divisibility rules
    'vị trí lẻ',           # "odd position" → digit position rules
    'lưới ô vuông',        # "grid of squares" → grid/matrix problem
    '4x4',                 # grid dimensions
    
    # modular
    'giá trị trung bình',  # "average value" when combined with mode
    '1,5 lần mode',        # "1.5 times the mode"
    # multi_variable
    'X phút',           # unknown in word problem
    'tích vô hướng',    # dot product → vectors
    'vectơ',            # vector
    'chu vi của hình chữ nhật',  # perimeter optimization
    
    # system_of_eq
    'tỷ lệ thuận',      # proportional → ratio setup
    'giải biến',        # "solve for variable"
    'đồng thời',        # "simultaneously" → dead giveaway for systems
    'x và y',           # two unknowns named together
    'f(x)',        # function notation
    'f(7)',        # function evaluation  
    '(0,',         # decimal percentage chains
    'khai triển',  # bracket expansion
    'mở rộng',    # "expand"
    'hạng nhất',  # first class → percentage problems
    '3x ',          # algebraic terms like 3x, 5x
    '5x ',
    '8x ',
    'nhân chéo',    # "cross multiply"
    'mẫu số chung', # "common denominator"
    'số bình phương',  # perfect squares
    'số phức',         # complex numbers
    'ước của',         # divisors/factors
    'Giả sử ',         # "assume/suppose" — variable setup
    'viết lại',        # "rewrite" — algebraic manipulation
    'lũy thừa',
    'liên tiếp',    # consecutive numbers — usually algebra setup
    'bổ sung',      # supplement/complement — geometry algebra
    'phần bù',      # complement angle
    'đơn giản hóa', # "simplify" — algebraic manipulation
    'giải phương trình',  # "solve the equation"
    'bắt tay',            # handshake — combinatorics
    'hai lần',            # "twice/two times" — double counting logicc
    'công thức',    # "formula" — plugging into a formula
    'căn bậc hai',   # square root — plain text version
    'kết hợp',       # combinations — combinatorics
    'bội số của',    # multiples of — number theory
    'làm tròn',      # rounding
    'đa thức',       # polynomial
    'chia đều',      # "divide equally" — usually multi-step
    'PEMDAS',
    'từ trái sang phải',  # left to right — order of operations
    'thứ tự thực hiện',   # order of operations
    'thay x=',      # substitution with specific variable
    'thay x =',
]

COMPLEX_PATTERNS_GSM = [ '[asy]',
    '\\begin{align',
    '\\log', '\\sqrt', '\\binom',
    '\\sum', '\\prod', '\\lim', '\\int', '\\infty',
    '\\text{', 'H_{', '\\begin{pmatrix}',
    '\\pmod', '\\equiv',
    '\\sin', '\\cos', '\\tan', '\\pi',
    '\\alpha', '\\beta', '\\theta', '\\nabla', '\\mathbf',
    '\\begin', '\\multicolumn', '\\cline', '\\overline',
    '\\lfloor', '\\rfloor',
    '\\dbinom', '\\choose', '\\omega', '\\gcd', '\\mod',
    '\\cap', '\\cup', '\\displaystyle', '\\Rightarrow',
    '\\Diamond', '\\spadesuit', '\\star',
    '\\left', '<=ft', '\\right',
    '\\clubsuit', '\\diamondsuit', '\\odot',
    '\\lceil', '\\rceil', '\\overrightarrow', '\\operatorname',
    '\\Delta', '\\Theta', '\\implies', '\\oplus',
    '\\mathrm', '\\textrm', '\\rm', '\\heartsuit',
    '\\arcsin', '\\phi', '\\delta', '\\partial',
    '\\stackrel', '\\cancel', '\\frown', '\\bold',
    '\\underline', '\\mbox', '\\mathcal', '\\bmod',
    '\\det', '\\mathbb', '\\textnormal', '\\color',
    '\\in','\\text','\\cotang',
    '\\widehat',
    '\\vdots',
    '\\mapsto',
    '\\]',   # catches the \[ ... \] display math blocks
    '\\[',]

def is_too_complex(record):
    response = str(record['response_vi'])
    return any(pattern in response for pattern in COMPLEX_PATTERNS)
    
def is_too_complex_gsm(record):
    response = str(record['response_vi'])
    return any(pattern in response for pattern in COMPLEX_PATTERNS_GSM)

def is_english(text):
    text = str(text)
    # Vietnamese-specific characters
    vietnamese_chars = set('àáâãèéêìíòóôõùúýăđơưạảấầẩẫậắằẳẵặẹẻẽếềểễệỉịọỏốồổỗộớờởỡợụủứừửữựỳỵỷỹ'
                          'ÀÁÂÃÈÉÊÌÍÒÓÔÕÙÚÝĂĐƠƯẠẢẤẦẨẪẬẮẰẲẴẶẸẺẼẾỀỂỄỆỈỊỌỎỐỒỔỖỘỚỜỞỠỢỤỦỨỪỬỮỰỲỴỶỸ')
    has_vietnamese = any(c in vietnamese_chars for c in text)
    if has_vietnamese:
        return False  # definitely Vietnamese
    
    # No Vietnamese chars → check if it's actually English
    words = text.split()
    if not words:
        return False
    english_chars = sum(1 for c in text if ord(c) < 128 and c.isalpha())
    total_chars = sum(1 for c in text if c.isalpha())
    return (english_chars / total_chars) > 0.8 if total_chars > 0 else False

In [ ]:
import json

with open("/kaggle/input/datasets/kimanh2002/dataset-math/train.json", "r") as f:
    data = json.load(f)

print(type(data))
print(len(data))
print(data[0])

In [ ]:
df = pd.DataFrame(data)
print(df.shape)
print(df.columns.tolist())
print(df['type'].value_counts())

In [ ]:
# Load and split
df = df[["original_question_vi","query_vi", "response_vi", "type"]]
records = df.to_dict(orient='records')

for r in records:
    r['response_vi'] = fix_boxed(str(r['response_vi']))

# Drop English records before splitting
before = len(records)
records = [r for r in records if not is_english(r['query_vi']) and not is_english(r['response_vi'])]
print(f"Dropped {before - len(records)} English records")

gsm_records = [r for r in records if 'GSM' in str(r['type'])]
math_records = [r for r in records if 'MATH' in str(r['type'])]
print(f"GSM: {len(gsm_records)}, MATH: {len(math_records)}")

In [ ]:
math_records = [r for r in records if 'MATH' in str(r['type'])]

# --- MATH ---
math_records = [r for r in math_records if not is_unsolvable(r)]
math_records = [r for r in math_records if has_answer_marker(r)]
boxed_values = [(r, extract_boxed(str(r['response_vi']))) for r in math_records]
math_records = [r for r, v in boxed_values if is_salvageable(r, v)]
math_records = [r for r in math_records if not is_too_complex(r)]

for r in math_records:
    r['response_vi'] = strip_dollar(str(r['response_vi']))
    r['response_vi'] = clean_cdot(str(r['response_vi']))
    r['response_vi'] = clean_latex(str(r['response_vi']))
    r['response_vi'] = convert_frac(str(r['response_vi']))

# Drop anything convert_frac couldn't clean
math_records = [r for r in math_records if '\\frac' not in str(r['response_vi'])]

print(f"MATH records final: {len(math_records)}")

# Verify no complex LaTeX remaining
latex_commands = []
for r in math_records:
    commands = re.findall(r'\\[a-zA-Z]+', str(r['response_vi']))
    latex_commands.extend(commands)

from collections import Counter
print("\nRemaining LaTeX commands:")
for cmd, count in sorted(Counter(latex_commands).items(), key=lambda x: -x[1])[:20]:
    print(f"{cmd}: {count}")

In [ ]:
gsm_records = [r for r in records if 'GSM' in str(r['type'])]

In [ ]:
import re
import random

def normalize_answer_marker(text):
    # Replace "Câu trả lời là: X" → "Đáp án là: X"
    text = re.sub(r'Câu trả lời là:\s*', 'Đáp án là: ', text)
    # Remove \boxed {X} or \boxed{X} (handle optional space)
    text = re.sub(r'\\boxed\s*\{([^}]+)\}', r'\1', text)
    return text

def remove_duplicate_answer(text):
    match = re.search(r'Đáp án là:\s*.+$', text.strip())
    if not match:
        return text
    before = text[:match.start()]
    before = re.sub(r'Đáp án là:\s*\S+\s*(?=[.\n]|$)', '', before)
    before = re.sub(r'Câu trả lời là:\s*\S+\s*(?=[.\n]|$)', '', before)
    # Strip any dangling partial unicode word at the end of `before`
    before = re.sub(r'\s+\S{1,5}$', ' ', before).rstrip()
    return (before + ' ' + match.group(0)).strip()

# Apply to both
for r in math_records:
    r['response_vi'] = normalize_answer_marker(str(r['response_vi']))
    r['response_vi'] = remove_duplicate_answer(str(r['response_vi']))

for r in gsm_records:
    r['response_vi'] = remove_duplicate_answer(str(r['response_vi']))

def clean_gsm_artifacts(text):
    # Remove ####N or #### N (GSM8K answer marker) and <<...>> (calculator annotations)
    text = re.sub(r'<<[^>]+>>', '', text)
    text = re.sub(r'####\s*\S+', '', text)
    return text.strip()

# --- GSM ---
gsm_records = [r for r in gsm_records if not is_unsolvable(r)]
gsm_records = [r for r in gsm_records if has_answer_marker(r)]
gsm_records = [r for r in gsm_records if not is_too_complex_gsm(r)]

# Clean
for r in gsm_records:
    r['response_vi'] = clean_gsm_artifacts(str(r['response_vi']))
    r['response_vi'] = strip_dollar(str(r['response_vi']))
    r['response_vi'] = clean_cdot(str(r['response_vi']))
    r['response_vi'] = clean_latex(str(r['response_vi']))
    r['response_vi'] = convert_frac(str(r['response_vi']))

gsm_records = [r for r in gsm_records if '\\frac' not in str(r['response_vi'])]
print(f"GSM records final: {len(gsm_records)}")

In [ ]:
random.seed(42)
all_records = math_records + gsm_records
random.shuffle(all_records)
print(f"Total records: {len(all_records)}")

In [ ]:
for r in all_records:
    r['response_vi'] = str(r['response_vi']).replace('Câu trả lời là', 'Đáp án là')

In [ ]:
from transformers import AutoTokenizer
import numpy as np

tokenizer = AutoTokenizer.from_pretrained("/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese")

token_lengths = [
    len(tokenizer.encode(f"{r['query_vi']} {r['response_vi']}"))
    for r in all_records
]
MAX_TOKENS = 372
filtered_records = [r for r, l in zip(all_records, token_lengths) if l <= MAX_TOKENS]
print(f"Kept {len(filtered_records)} / {len(all_records)} records ({100*len(filtered_records)/len(all_records):.1f}%)")

In [ ]:
# Full filtered df
df_filtered = pd.DataFrame(filtered_records)
print(f"df_filtered: {len(df_filtered)} records")
print(df_filtered['type'].value_counts())


In [ ]:
import os
os.makedirs('/kaggle/working', exist_ok=True)

df_filtered.to_csv('/kaggle/working/filter_latex_and_hard_math.csv', index=False)

# Curriculum learning data

In [ ]:
import numpy as np # linear algebra
import pandas as pd
import re
df = pd.read_csv("/kaggle/working/filter_latex_and_hard_math.csv")



# AND has explicit arithmetic in response
pattern_mask = df['response_vi'].str.contains(
    r'\d+\s*[\+\-\*\/]\s*\d+',
    regex=True
)

simple_df = df[pattern_mask].reset_index(drop=True)
print(f"Total: {len(simple_df)}")
simple_df[['query_vi', 'response_vi', 'type']]

In [ ]:
def extract_arithmetic(text):
    parts = re.findall(r'[\d\.\,\/\*\+\-\×\÷\s]+\=[\d\.\,\s]+', text)
    parts = [p.strip().rstrip('.') for p in parts]
    parts = [re.sub(r'^[,\.\s]+', '', p).strip() for p in parts]
    parts = [p for p in parts if p and not re.match(r'^[\d\.\,\s]+=[\d\.\,\s]+$', p)]

    deduped = []
    for p in parts:
        if not deduped or p != deduped[-1]:
            deduped.append(p)
    parts = deduped

    merged = []
    for p in parts:
        if merged and re.match(r'^[\+\-\*\/\×\÷]', p):
            merged[-1] += ' ' + p
        else:
            merged.append(p)

    def normalize_vn_number(s):
        s = s.strip()
        # If it matches pattern like 1.000 or 17.000 (thousands separator), remove the dot
        s = re.sub(r'(\d)\.(\d{3})(?!\d)', r'\1\2', s)
        # Replace comma decimal separator with dot
        s = s.replace(',', '.')
        return s

    def parse_vn_number(s):
        return float(normalize_vn_number(s))

    validated = []
    for p in merged:
        if '=' not in p:
            continue
        lhs, rhs = p.split('=', 1)
        try:
            expr = normalize_vn_number(lhs.strip()).replace('×', '*').replace('÷', '/')
            expected = eval(expr)
            expected = int(expected) if expected == int(expected) else round(expected, 2)
            rhs_val = parse_vn_number(rhs.strip())
            rhs_normalized = int(rhs_val) if rhs_val == int(rhs_val) else round(rhs_val, 2)
            if expected == rhs_normalized:
                validated.append(p)
            else:
                return ''
        except:
            return ''

    if not validated:
        return ''

    answer = re.findall(r'[Đđ]áp án[^\d]*(\d+)', text)
    result = ' | '.join(validated)
    if answer:
        result += f' | Đáp án là: {answer[-1]}'
    return result
    
def is_clean(text):
    parts = text.split('|')
    for p in parts:
        p = p.strip()
        if re.match(r'^[\+\-\*\/\×\÷\=]', p):
            return False
        if re.match(r'^\d+\s*=', p):
            return False
        if re.match(r'^[,\.\s]', p):
            return False
        if p == '':
            return False
        if 'Đáp án' not in p:
            # check both sides of = have numbers
            if '=' in p:
                sides = p.split('=')
                if not re.search(r'\d', sides[0]):  # nothing before =
                    return False
                if not re.search(r'\d', sides[1]):  # nothing after =
                    return False
    return True

def is_consistent_answer(text):
    parts = [p.strip() for p in text.split('|')]
    
    # Find "Đáp án là" part
    dap_an_part = None
    for p in parts:
        if 'Đáp án là' in p or 'Đáp án la' in p:
            dap_an_part = p
            break
    
    if not dap_an_part:
        return True  # no answer to check against, keep it
    
    # Extract the answer number
    dap_an_match = re.search(r'(\d[\d\.\,]*)', dap_an_part)
    if not dap_an_match:
        return True
    dap_an_num = dap_an_match.group(1).replace('.', '').replace(',', '')

    # Get the part just before "Đáp án là"
    pre_parts = [p for p in parts if 'Đáp án' not in p]
    if not pre_parts:
        return True
    
    last_expr = pre_parts[-1]
    
    # Extract the result (right side of =)
    eq_match = re.search(r'=\s*([\d\.\,]+)', last_expr)
    if not eq_match:
        return True
    last_num = eq_match.group(1).replace('.', '').replace(',', '')

    return last_num == dap_an_num

## Stage 1 augmentation

In [ ]:
simple_df['response_clean'] = simple_df['response_vi'].apply(extract_arithmetic)
stage1_df = simple_df[['query_vi', 'response_clean','type']].rename(columns={'response_clean': 'response_vi'})
stage1_df = stage1_df[stage1_df['response_vi'].apply(is_clean)].reset_index(drop=True)
stage1_df = stage1_df[stage1_df['response_vi'].apply(is_consistent_answer)].reset_index(drop=True)
stage1_df.to_csv("stage1_arithmetic.csv", index=False)
print(f"Stage 1: {len(stage1_df)}")

In [ ]:
stage2_df = df[~df.index.isin(stage1_df.index)].reset_index(drop=True)
print(f"Remaining: {len(stage2_df)}")

In [ ]:
import re
import random
from fractions import Fraction
def scale_expression(expr, running_subs, scale_factor):
    def replace_num(match):
        token = match.group(0)
        start = match.start()
        # Skip if part of a fraction (preceded or followed by /)
        context = expr[max(0, start-1):match.end()+1]
        if re.search(r'[\d][\/]' + re.escape(token), context) or \
           re.search(re.escape(token) + r'[\/][\d]', context):
            return token
        if token in running_subs:
            return format_vn_number(running_subs[token])
        try:
            val = float(normalize_vn_number(token))
            new_val = val * scale_factor
            new_val = int(new_val) if new_val == int(new_val) else round(new_val, 2)
            return format_vn_number(new_val)
        except:
            return token
    return re.sub(r'\d+(?:[.,]\d+)?', replace_num, expr)

def substitute_numbers(query, answer, scale_factor=None, random_seed=None):
    if random_seed is not None:
        random.seed(random_seed)
    if scale_factor is None:
        scale_factor = random.choice([2, 3, 4, 5, 0.5])

    query_nums = re.findall(r'\d+(?:[.,]\d+)?', query)
    if not query_nums:
        return None, None

    substitutions = {}
    for n in query_nums:
        start = query.find(n)
        # Skip percentages
        if query[start + len(n):start + len(n) + 1] == '%':
            continue
        # Skip fractions (preceded or followed by /)
        if (start > 0 and query[start-1] == '/') or \
           (start + len(n) < len(query) and query[start + len(n)] == '/'):
            continue
        normalized = float(normalize_vn_number(n))
        new_val = normalized * scale_factor
        new_val = int(new_val) if new_val == int(new_val) else round(new_val, 2)
        substitutions[n] = new_val

    if not substitutions:
        return None, None

    new_query = query
    for orig in sorted(substitutions.keys(), key=len, reverse=True):
        if ',' in orig:
            new_str = format_vn_number(substitutions[orig])
        else:
            new_str = str(substitutions[orig])
        new_query = new_query.replace(orig, new_str, 1)

    new_answer = recompute_chain(answer, substitutions, scale_factor)
    if new_answer is None:
        return None, None

    return new_query, new_answer


def normalize_vn_number(s):
    s = s.strip()
    s = re.sub(r'(\d)\.(\d{3})(?!\d)', r'\1\2', s)
    s = s.replace(',', '.')
    return s

def format_vn_number(val):
    """Format a number back to Vietnamese style (comma as decimal sep)."""
    if val == int(val):
        return str(int(val))
    else:
        return str(round(val, 2)).replace('.', ',')

def recompute_chain(answer, substitutions, scale_factor):
    """
    Recompute each step in the answer chain by scaling the result of each step.
    Simpler approach: just scale all numbers in the answer chain too,
    then re-eval each LHS and patch the RHS.
    """
    # Strip the "Đáp án là" part
    dap_an_match = re.search(r'\|\s*Đáp án là:\s*(\d+(?:[.,]\d+)?)', answer)
    
    parts = [p.strip() for p in answer.split('|')]
    parts = [p for p in parts if 'Đáp án' not in p]

    new_parts = []
    # We'll carry a running substitution map that grows as we compute new results
    running_subs = dict(substitutions)

    for p in parts:
        if '=' not in p:
            continue
        lhs, rhs = p.split('=', 1)

        # Scale all numbers in lhs using running_subs, then eval
        new_lhs = scale_expression(lhs.strip(), running_subs, scale_factor)
        if new_lhs is None:
            return None

        try:
            expr = normalize_vn_number(new_lhs).replace('×', '*').replace('÷', '/')
            result = eval(expr)
            result = int(result) if result == int(result) else round(result, 2)
        except:
            return None

        # Add old rhs -> new result to running_subs so downstream steps pick it up
        old_rhs = rhs.strip()
        running_subs[old_rhs] = result

        new_rhs = format_vn_number(result)
        new_parts.append(f"{new_lhs} = {new_rhs}")

    if not new_parts:
        return None

    # Get final answer from last step's RHS
    last_result = new_parts[-1].split('=')[-1].strip()
    # Strip comma formatting for "Đáp án là"
    last_num = last_result.replace(',', '')
    result_str = ' | '.join(new_parts)
    result_str += f' | Đáp án là: {last_num}'
    return result_str


def is_valid_augmentation(new_query, new_answer):
    """Check augmented query and answer are internally consistent."""
    query_nums = set(re.findall(r'\d+(?:[.,]\d+)?', new_query))
    
    parts = [p.strip() for p in new_answer.split('|')]
    parts = [p for p in parts if 'Đáp án' not in p]
    
    known = set(query_nums)
    
    for p in parts:
        if '=' not in p:
            continue
        lhs, rhs = p.split('=', 1)
        lhs_nums = re.findall(r'\d+(?:[.,]\d+)?', lhs)
        for n in lhs_nums:
            if n not in known:
                return False
        known.update(re.findall(r'\d+(?:[.,]\d+)?', rhs))
    
    return True

def augment_dataset(df, n_augments=2, scale_factors=None, random_seed=42):
    if scale_factors is None:
        scale_factors = [2, 3, 5]
    random.seed(random_seed)
    augmented_rows = []
    for row in df.itertuples():
        for i, sf in enumerate(scale_factors[:n_augments]):
            new_query, new_answer = substitute_numbers(
                row.query_vi, row.response_vi,
                scale_factor=sf,
                random_seed=random_seed + i
            )
            if new_query is None:
                continue
            if not is_clean(new_answer):
                continue
            if not is_consistent_answer(new_answer):
                continue
            if not is_valid_augmentation(new_query, new_answer):
                continue
            augmented_rows.append({
                'query_vi': new_query,
                'response_vi': new_answer,
                'type': row.type
            })
    aug_df = pd.DataFrame(augmented_rows)
    result_df = pd.concat([df, aug_df], ignore_index=True).drop_duplicates(subset='query_vi')
    print(f"Original: {len(df)}")
    print(f"Augmented: {len(aug_df)}")
    print(f"Total: {len(result_df)}")
    return result_df

In [ ]:
def augment_additive(df, n_augments=5, max_offset=50, random_seed=42):
    random.seed(random_seed)
    augmented_rows = []

    for row in df.itertuples():
        parts = [p.strip() for p in row.response_vi.split('|')]
        parts = [p for p in parts if 'Đáp án' not in p]

        # Only handle single-step problems
        if len(parts) != 1:
            continue

        p = parts[0]
        if '=' not in p:
            continue

        lhs, rhs = p.split('=', 1)

        # Only handle + or - operations
        if not re.search(r'[\+\-]', lhs) or re.search(r'[\*\/\×\÷]', lhs):
            continue

        # Extract operator
        op_match = re.search(r'([\+\-])', lhs)
        if not op_match:
            continue
        op = op_match.group(1)

        # Extract the two operands
        operands = re.findall(r'\d+(?:[.,]\d+)?', lhs)
        if len(operands) != 2:
            continue

        for i in range(n_augments):
            # Randomly shift first operand
            try:
                a = float(normalize_vn_number(operands[0]))
                b = float(normalize_vn_number(operands[1]))
            except:
                continue

            offset = random.randint(1, max_offset)
            sign = random.choice([-1, 1])
            new_a = a + offset * sign

            if new_a <= 0:
                continue

            new_a = int(new_a) if new_a == int(new_a) else round(new_a, 2)

            if op == '+':
                new_result = new_a + b
            else:
                new_result = new_a - b

            if new_result <= 0:
                continue

            new_result = int(new_result) if new_result == int(new_result) else round(new_result, 2)

            # Substitute new_a into query
            new_query = row.query_vi.replace(operands[0], format_vn_number(new_a), 1)
            if new_query == row.query_vi:
                continue

            new_lhs = lhs.replace(operands[0], format_vn_number(new_a), 1)
            new_answer = f"{new_lhs.strip()} = {format_vn_number(new_result)} | Đáp án là: {int(new_result) if new_result == int(new_result) else new_result}"

            if not is_clean(new_answer):
                continue
            if not is_consistent_answer(new_answer):
                continue
            if not is_valid_augmentation(new_query, new_answer):
                continue

            augmented_rows.append({
                'query_vi': new_query,
                'response_vi': new_answer,
                'type': row.type
            })

    aug_df = pd.DataFrame(augmented_rows)
    aug_df = aug_df.drop_duplicates(subset='query_vi')
    aug_df = aug_df[~aug_df['query_vi'].isin(df['query_vi'])]

    print(f"Additive augmented: {len(aug_df)}")
    return aug_df

In [ ]:
stage1_df_dedup = stage1_df.drop_duplicates(subset='query_vi').reset_index(drop=True)
print(f"Deduped original: {len(stage1_df_dedup)}")

stage1_augmented = augment_dataset(
    stage1_df_dedup, 
    n_augments=10, 
    scale_factors=[2, 3, 4, 5, 6, 7, 8, 10, 1.5, 2.5]
)

additive_aug = augment_additive(
    stage1_df_dedup, 
    n_augments=10, 
    max_offset=50
)

stage1_augmented = pd.concat(
    [stage1_augmented, additive_aug], 
    ignore_index=True
).drop_duplicates(subset='query_vi')

print(f"Final total: {len(stage1_augmented)}")

## Stage 2 augmentation

In [ ]:
def cross_augment_by_anchor_gsm_only(df, target_types=['GSM_Rephrased', 'GSM_AnsAug'], pairs_per_anchor=2, random_seed=42):
    random.seed(random_seed)
    subset = df[df['type'].isin(target_types)].copy()
    new_rows = []
    grouped = subset.groupby('original_question_vi')

    for anchor, group in grouped:
        rephrased = group[group['type'] == 'GSM_Rephrased']
        ansaug    = group[group['type'] == 'GSM_AnsAug']

        if len(rephrased) == 0 or len(ansaug) == 0:
            continue

        rephrased_queries   = rephrased['query_vi'].unique().tolist()
        rephrased_responses = rephrased['response_vi'].unique().tolist()
        ansaug_queries      = ansaug['query_vi'].unique().tolist()
        ansaug_responses    = ansaug['response_vi'].unique().tolist()
        existing_pairs      = set(zip(group['query_vi'], group['response_vi']))

        for direction, q_pool, r_pool, t in [
            ('rephrase->ansaug', rephrased_queries, ansaug_responses, 'GSM_Rephrased'),
            ('ansaug->rephrase', ansaug_queries, rephrased_responses, 'GSM_AnsAug'),
        ]:
            added, attempts = 0, 0
            while added < pairs_per_anchor and attempts < pairs_per_anchor * 5:
                q = random.choice(q_pool)
                r = random.choice(r_pool)
                attempts += 1
                if (q, r) not in existing_pairs:
                    existing_pairs.add((q, r))
                    new_rows.append({'type': t, 'query_vi': q, 'response_vi': r})
                    added += 1

    augmented_df = pd.DataFrame(new_rows)
    result = pd.concat([df[['query_vi', 'response_vi', 'type']], augmented_df], ignore_index=True).drop_duplicates(
        subset=['query_vi', 'response_vi']
    )

    print(f"Original:  {len(df):,}")
    print(f"Added:     {len(augmented_df):,}")
    print(f"Total:     {len(result):,}")
    return result

In [ ]:

# Version 3: Drop MATH types first, then augment with pairs_per_anchor=1
math_types = [t for t in stage2_df['type'].unique() if 'MATH' in t.upper()]
print(f"Dropping MATH types: {math_types}")

stage2_no_math = stage2_df[~stage2_df['type'].isin(math_types)].copy()
print(f"Rows before drop: {len(stage2_df):,}")
print(f"Rows after drop:  {len(stage2_no_math):,}")

stage2_no_math_augmented = cross_augment_by_anchor_gsm_only(
    stage2_no_math,
    target_types=['GSM_Rephrased', 'GSM_AnsAug'],
    pairs_per_anchor=1,
    random_seed=42
)

In [ ]:
def trim_to_token_budget(df, budget=8_300_000, text_columns=['query_vi', 'response_vi'], label=""):
    lengths = df[text_columns].apply(
        lambda col: col.dropna().astype(str).apply(
            lambda t: len(tokenizer.encode(t, add_special_tokens=False))
        )
    ).sum(axis=1)
    
    df = df.copy()
    df['_token_len'] = lengths
    df_sorted = df.sort_values('_token_len', ascending=True).reset_index(drop=True)
    
    cumsum = df_sorted['_token_len'].cumsum()
    cutoff = (cumsum <= budget).sum()
    
    result = df_sorted.iloc[:cutoff].drop(columns='_token_len')
    actual_tokens = cumsum.iloc[cutoff - 1]
    print(f"{label}: {len(result):,} rows, {actual_tokens:,} tokens")
    return result


stage2_no_math_trimmed      = trim_to_token_budget(stage2_no_math_augmented, label="stage2_no_math_augmented")
# Final df
stage1_df.to_csv("stage1_arithmetic.csv")
stage1_augmented.to_csv("stage1_augmented_arithmetic.csv")
stage2_no_math_trimmed.to_csv("stage2_augmented_gsm_only.csv", index=False)

# Training

# Fine-tune GPT-2 Vietnamese for Math Word Problems

**Goal:** fine-tune `NlpHUST/gpt2-vietnamese` to solve Vietnamese math word problems.

**Inputs:** `train.json`, `valid.json`, and the local GPT-2 model folder from Kaggle Input.

**Pipeline:** load data → SFT training → generate validation outputs → evaluate by relative error.

**Rules:** Internet OFF, no extra data/API/LLM, total runtime ≤ 3 hours.


In [ ]:
import os, sys, json, math, time, re, random, hashlib, inspect
from pathlib import Path
from dataclasses import dataclass
from typing import Dict, List

import torch
from torch.utils.data import Dataset
from tqdm.auto import tqdm
from transformers import AutoModelForCausalLM, AutoTokenizer, Trainer, TrainingArguments
from peft import get_peft_model, LoraConfig, TaskType,PeftModel, PeftConfig
import peft
import os
print("PEFT:", peft.__version__)

print("Torch:", torch.__version__)
print("CUDA:", torch.cuda.is_available(), "| GPU count:", torch.cuda.device_count())
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(i, torch.cuda.get_device_name(i))


In [ ]:
from pathlib import Path
stage1_csv = Path("/kaggle/working/stage1_augmented_arithmetic.csv")
stage2_csv = Path("/kaggle/working/stage2_augmented_gsm_only.csv")   # ← your 48k CSV path


In [ ]:
# ============================================================
# 1. Config
# ============================================================
def first_existing(*paths) -> Path:
    for p in map(Path, paths):
        if p.exists():
            return p
    raise FileNotFoundError("Không tìm thấy path nào: " + " | ".join(map(str, paths)))

DATA_DIR = first_existing(
    "/kaggle/input/datasets/kimanh2002/dataset-math",
    "/kaggle/input/dataset-math",
)
MODEL_NAME = str(first_existing(
    "/kaggle/input/datasets/kimanh2002/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese",
    "/kaggle/input/nlphustgpt2-vietnamese/gpt2-vietnamese",
))

TRAIN_FILE = DATA_DIR / "train.json"
VALID_FILE = DATA_DIR / "valid.json"

PROMPT_TEMPLATE = "Câu hỏi: {q}\nLời giải: "
SAFE_EOS_ID = 50256

OUTPUT_DIR = Path("/kaggle/working/gpt2_math_ckpt")
VALID_OUTPUT_PATH = Path("/kaggle/working/valid_output.json")
VALID_REPORT_PATH = Path("/kaggle/working/valid_report.json")
BASELINE_OUTPUT_PATH = Path("/kaggle/working/baseline_valid_output.json")
BASELINE_REPORT_PATH = Path("/kaggle/working/baseline_valid_report.json")

MAX_TRAIN_SAMPLES = None
MAX_VALID_SAMPLES = None

TRAIN_BOTH_ADAPTER = False
# Stage 1 — easy 10k data
STAGE1_EPOCHS = 2
STAGE1_LR = 5e-3
STAGE1_PER_DEVICE_BATCH_SIZE = 16
STAGE1_GRAD_ACCUM = 2          # effective batch size stays the same (8×4 = 32)  

STAGE1_WARMUP_RATIO = 0.03
STAGE1_WEIGHT_DECAY = 0.01
# Stage 2 — full 48k data

STAGE2_EPOCHS = 2
STAGE2_LR = 5e-3       # lower LR to preserve Stage 1 learning

STAGE2_PER_DEVICE_BATCH_SIZE = 16
STAGE2_GRAD_ACCUM = 1          # effective batch size stays the same (8×4 = 32)    

STAGE2_WARMUP_RATIO = 0.0
STAGE2_WEIGHT_DECAY = 0.01

MAX_LENGTH = 380



SEED = 42


MAX_NEW_TOKENS = 256
RUN_BASELINE_FIRST = False

def seed_everything(seed=42):
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(SEED)

print("TRAIN_FILE:", TRAIN_FILE)
print("VALID_FILE:", VALID_FILE)
print("MODEL_NAME:", MODEL_NAME)


In [ ]:
# ============================================================
# 2. Data loading
# ============================================================
def load_records(path: str | Path) -> list:
    p = Path(path)
    with p.open("r", encoding="utf-8") as f:
        head = f.read(1)
        f.seek(0)
        return json.load(f) if head == "[" else [json.loads(line) for line in f if line.strip()]

def sha256_file(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

def sha256_dir(dir_path: Path, suffixes=(".bin", ".safetensors", ".json", ".txt", ".model")) -> str:
    h = hashlib.sha256()
    for p in sorted(x for x in dir_path.rglob("*") if x.is_file() and x.suffix in suffixes):
        h.update(p.relative_to(dir_path).as_posix().encode() + b"\0")
        h.update(sha256_file(p).encode() + b"\0")
    return h.hexdigest()



In [ ]:
# ============================================================
# 3. SFT dataset and collator
# ============================================================
class SFTDataset(Dataset):
    """Tokenize (prompt, response), mask loss on prompt + padding."""

    def __init__(self, records, tokenizer, max_length: int):
        self.records = records
        self.tok = tokenizer
        self.max_length = max_length

    def __len__(self) -> int:
        return len(self.records)

    def __getitem__(self, i: int) -> Dict[str, List[int]]:
        rec = self.records[i]
        prompt = PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())
        response = rec["response_vi"].strip()

        p_ids = self.tok(prompt, add_special_tokens=False)["input_ids"]
        r_ids = self.tok(response, add_special_tokens=False)["input_ids"] + [SAFE_EOS_ID]

        ids = (p_ids + r_ids)[: self.max_length]
        labels = ([-100] * len(p_ids) + r_ids)[: self.max_length]

        # Defensive clamp: never feed out-of-range ids to embedding layer.
        ids = [min(t, SAFE_EOS_ID) for t in ids]
        labels = [(-100 if t == -100 else min(t, SAFE_EOS_ID)) for t in labels]

        return {
            "input_ids": ids,
            "labels": labels,
            "attention_mask": [1] * len(ids),
        }

@dataclass
class PadCollator:
    pad_id: int = SAFE_EOS_ID

    def __call__(self, batch):
        maxlen = max(len(x["input_ids"]) for x in batch)
        out = {"input_ids": [], "attention_mask": [], "labels": []}
        for x in batch:
            n = len(x["input_ids"])
            pad = maxlen - n
            out["input_ids"].append(x["input_ids"] + [self.pad_id] * pad)
            out["attention_mask"].append(x["attention_mask"] + [0] * pad)
            out["labels"].append(x["labels"] + [-100] * pad)
        return {k: torch.tensor(v, dtype=torch.long) for k, v in out.items()}


In [ ]:
# ============================================================
# 4. Evaluation utilities
# ============================================================
VI_ANCHORS = [
    r"Câu trả lời là\s*[:：]?",
    r"Đáp án là\s*[:：]?",
    r"Đáp án\s*[:：]",
]

EN_ANCHORS = [
    r"The answer is\s*[:：]?",
    r"####",
]

BOXED_RE = re.compile(r"\\boxed\{([^{}]*(?:\{[^{}]*\}[^{}]*)*)\}")
SAFE_NS = {"sqrt": math.sqrt, "pi": math.pi}


def extract_answer(text: str | None, anchors: list[str]) -> str | None:
    """Extract final answer by anchors. First anchor wins; fallback to last \\boxed{}."""
    if not text:
        return None

    for anc in anchors:
        m = re.search(anc, text)
        if m:
            tail = text[m.end():].strip()
            tail = tail.split("\n")[0]
            return tail.strip().rstrip(".。、,")

    boxes = BOXED_RE.findall(text)
    if boxes:
        return boxes[-1].strip()

    return None


def extract_gold(rec: dict) -> str | None:
    """Gold answer is read only from gold records, not from prediction files."""
    return extract_answer(rec.get("response_vi"), VI_ANCHORS)


def extract_pred(rec: dict) -> str | None:
    """Predicted answer is read only from model_output."""
    return extract_answer(rec.get("model_output"), VI_ANCHORS + EN_ANCHORS)


def parse_number(s: str | None) -> float | None:
    """Best-effort parser for finite scalar numeric answers."""
    if s is None:
        return None

    t = s.strip()
    if not t:
        return None

    # Pure Vietnamese decimal: 1,5 -> 1.5
    if re.fullmatch(r"-?\d+,\d+", t):
        try:
            return float(t.replace(",", "."))
        except ValueError:
            return None

    # Pure English decimal / integer
    if re.fullmatch(r"-?\d+(?:\.\d+)?(?:[eE][+-]?\d+)?", t):
        try:
            val = float(t)
            return val if math.isfinite(val) else None
        except ValueError:
            return None

    # Strip variable assignment: x = 5 -> 5
    m = re.match(r"^[A-Za-z_]\w*\s*=\s*(.+)$", t)
    if m:
        t = m.group(1).strip()

    # Reject tuples / intervals early
    if t.startswith("(") and t.endswith(")") and re.search(r"\d\s*,\s*\d", t):
        return None
    if t.startswith("[") and t.endswith("]"):
        return None

    # Strip wrappers
    for _ in range(3):
        new = re.sub(r"\\boxed\{((?:[^{}]|\{[^{}]*\})*)\}", r"(\1)", t)
        if new == t:
            break
        t = new

    t = re.sub(r"\\text\{[^}]*\}", "", t)
    t = re.sub(r"\\mathrm\{[^}]*\}", "", t)
    t = t.replace("$", "")

    for token in ("\\,", "\\!", "\\;", "\\ ", "\\left", "\\right"):
        t = t.replace(token, "")

    for token in ("\\cdot", "\\times"):
        t = t.replace(token, "*")

    # LaTeX fractions / roots
    t = re.sub(r"\\(?:d|t)?frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}", r"((\1)/(\2))", t)
    t = re.sub(r"\\sqrt\s*\{([^{}]+)\}", r"sqrt(\1)", t)
    t = re.sub(r"\\sqrt\s*(\d+(?:\.\d+)?)", r"sqrt(\1)", t)
    t = t.replace("\\pi", "pi")

    # Implicit multiplication
    t = re.sub(r"(\d)\s*(sqrt|pi|\()", r"\1*\2", t)
    t = re.sub(r"(\))\s*(sqrt|pi|\d)", r"\1*\2", t)
    t = re.sub(r"(pi)\s*(sqrt|pi|\d|\()", r"\1*\2", t)

    # Comma handling
    has_period = "." in t
    n_commas = t.count(",")

    if n_commas == 1 and not has_period and re.search(r"\d,\d", t):
        # Vietnamese decimal
        t = re.sub(r"(?<=\d),(?=\d)", ".", t)
    elif n_commas >= 1:
        # English thousands separator: 1,000 -> 1000
        t = re.sub(r"(?<=\d),(?=\d{3}\b)", "", t)

    t = re.sub(r"\s+", "", t)

    if not t:
        return None

    # Reject remaining tuples/lists
    if "," in t:
        return None

    # Only allow safe expression chars
    leftover = re.sub(r"sqrt|pi|\d|\.|\+|\-|\*|/|\(|\)|\^|e|E", "", t)
    if leftover:
        return None

    t = t.replace("^", "**")

    try:
        val = eval(t, {"__builtins__": {}}, SAFE_NS)
    except Exception:
        return None

    if isinstance(val, bool):
        return None

    if isinstance(val, (int, float)):
        val = float(val)
        return val if math.isfinite(val) else None

    return None


def rel_error(pred: float | None, gold: float | None) -> float | None:
    if pred is None or gold is None:
        return None

    denom = max(1.0, abs(gold))
    return abs(pred - gold) / denom


def score_one(re_val: float | None, extractable: bool) -> int:
    if not extractable:
        return 0
    if re_val is None:
        return 0
    if re_val <= 0.01:
        return 10
    if re_val <= 0.10:
        return 5
    if re_val <= 0.50:
        return 1
    return 0


def align_predictions_with_gold(pred_items: list[dict], gold_items: list[dict]) -> list[tuple[dict, dict]]:
    """
    Align predictions and gold records.

    If both sides have id, align by id.
    Otherwise, require the same length and align by order.
    """
    pred_has_id = all("id" in x for x in pred_items)
    gold_has_id = all("id" in x for x in gold_items)

    if pred_has_id and gold_has_id:
        pred_map = {str(x["id"]): x for x in pred_items}
        pairs = []

        missing = []
        for g in gold_items:
            gid = str(g["id"])
            if gid not in pred_map:
                missing.append(gid)
            else:
                pairs.append((pred_map[gid], g))

        if missing:
            raise ValueError(f"Prediction thiếu {len(missing)} id, ví dụ: {missing[:5]}")

        return pairs

    if len(pred_items) != len(gold_items):
        raise ValueError(
            f"Số lượng prediction ({len(pred_items)}) khác số lượng gold ({len(gold_items)})."
        )

    return list(zip(pred_items, gold_items))


def evaluate(pred_items: list[dict], gold_items: list[dict]) -> dict:
    pairs = align_predictions_with_gold(pred_items, gold_items)

    rows = []
    total = 0
    bucket10 = bucket5 = bucket1 = bucket0 = 0
    extractable = 0
    numeric_pairs = 0
    rel_errors = []

    for pred_rec, gold_rec in pairs:
        gold_ans = extract_gold(gold_rec)
        pred_ans = extract_pred(pred_rec)

        is_extractable = pred_ans is not None
        extractable += int(is_extractable)

        gold_num = parse_number(gold_ans)
        pred_num = parse_number(pred_ans)
        re_val = rel_error(pred_num, gold_num)

        if gold_num is not None and pred_num is not None and re_val is not None:
            numeric_pairs += 1
            rel_errors.append(re_val)

        s = score_one(re_val, is_extractable)
        total += s

        bucket10 += int(s == 10)
        bucket5 += int(s == 5)
        bucket1 += int(s == 1)
        bucket0 += int(s == 0)

        rows.append({
            "id": gold_rec.get("id", pred_rec.get("id")),
            "type": gold_rec.get("type") or pred_rec.get("type"),
            "gold_answer": gold_ans,
            "pred_answer": pred_ans,
            "gold_num": gold_num,
            "pred_num": pred_num,
            "rel_error": re_val,
            "extractable": is_extractable,
            "score": s,
        })

    n = len(rows)
    max_score = n * 10
    score_10 = total / n if n else 0.0

    return {
        "summary": {
            "n": n,
            "raw_score": total,
            "max_raw_score": max_score,
            "score_10": score_10,
            "score_pct": total / max_score if max_score else 0.0,
            "extractable": extractable,
            "numeric_pairs": numeric_pairs,
            "buckets": {
                "10": bucket10,
                "5": bucket5,
                "1": bucket1,
                "0": bucket0,
            },
            "rel_error_mean": sum(rel_errors) / len(rel_errors) if rel_errors else None,
        },
        "rows": rows,
    }


def save_evaluation_report(
    pred_path: str | Path,
    gold_records: list[dict],
    report_path: str | Path,
) -> dict:
    pred_path = Path(pred_path)
    report_path = Path(report_path)

    with pred_path.open("r", encoding="utf-8") as f:
        pred_items = json.load(f)

    result = evaluate(pred_items, gold_records)

    print(json.dumps(result["summary"], ensure_ascii=False, indent=2))

    with report_path.open("w", encoding="utf-8") as f:
        json.dump(result, f, ensure_ascii=False, indent=2)

    print(f"Wrote {report_path}")
    return result

In [ ]:
# ============================================================
# 5. Greedy generation
# ============================================================
def build_prompt(rec: dict) -> str:
    return PROMPT_TEMPLATE.format(q=rec["query_vi"].strip())

def generate_outputs(model_path_or_name: str | Path, records: list, output_path: str | Path, max_new_tokens: int = 256):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    print(f"Loading {model_path_or_name} on {device} ...", flush=True)

    tokenizer = AutoTokenizer.from_pretrained(model_path_or_name, local_files_only=True)
    tokenizer.pad_token_id = SAFE_EOS_ID
    tokenizer.eos_token_id = SAFE_EOS_ID

    dtype = torch.float16 if device == "cuda" else torch.float32
    model = AutoModelForCausalLM.from_pretrained(
        model_path_or_name, torch_dtype=dtype, local_files_only=True
    ).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()

    vocab_n = model.transformer.wte.num_embeddings
    outputs, t0 = [], time.time()

    with torch.inference_mode():
        for rec in tqdm(records, desc="Generating"):
            prompt = build_prompt(rec)
            enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=MAX_LENGTH).to(device)

            ids = enc["input_ids"].clamp(max=vocab_n - 1)
            gen = model.generate(
                input_ids=ids,
                attention_mask=enc.get("attention_mask"),
                max_new_tokens=max_new_tokens,
                do_sample=False,
                num_beams=4,
                no_repeat_ngram_size=4,
                repetition_penalty=1.15,
                pad_token_id=SAFE_EOS_ID,
                eos_token_id=SAFE_EOS_ID,
            )

            text = tokenizer.decode(gen[0, ids.shape[1]:], skip_special_tokens=True)
            outputs.append({
                "id": len(outputs),
                "query_vi": rec["query_vi"],
                "type": rec.get("type"),
                "model_output": text,
            })

    output_path = Path(output_path)
    with output_path.open("w", encoding="utf-8") as f:
        json.dump(outputs, f, ensure_ascii=False, indent=2)

    out_hash = sha256_file(output_path)
    Path(str(output_path) + ".sha256.txt").write_text(out_hash + "\n", encoding="utf-8")

    dt = time.time() - t0
    print(f"Wrote {output_path} | {dt/60:.2f} min | SHA256: {out_hash}")
    return outputs


In [ ]:
from pathlib import Path
import pandas as pd
import shutil
import peft


valid_records = load_records(VALID_FILE)
all_results = {}

# ============================================================
# Toggle approach here
# "continued"  — same adapter trained through both stages
# "freeze"      — merge stage1, fresh adapter for stage2
# ============================================================
FREEZE_STAGE_1_ADAPTER = True  # or "continued"

print(f"\n{'='*60}")
print(f"Training on: {stage1_csv.name} → {stage2_csv.name}")
print(f"FREEZING STAGE 1: {FREEZE_STAGE_1_ADAPTER}")
print(f"{'='*60}")

# Load stage 1 CSV
df = pd.read_csv(stage1_csv)
if "query_vi" not in df.columns:
    df = df.rename(columns={"question": "query_vi", "answer": "response_vi"})
train_records = df.to_dict("records")
print(f"Stage 1 records: {len(train_records)}")

# Load stage 2 CSV
df2 = pd.read_csv(stage2_csv)
if "query_vi" not in df2.columns:
    df2 = df2.rename(columns={"question": "query_vi", "answer": "response_vi"})
train_records_s2 = df2.to_dict("records")
print(f"Stage 2 records: {len(train_records_s2)}")

run_output_dir = Path("/kaggle/working/current_run")
run_output_dir.mkdir(parents=True, exist_ok=True)
stage1_merged_dir = Path("/kaggle/working/stage1_merged")

valid_output_path = run_output_dir / "valid_output.json"
valid_report_path = run_output_dir / "valid_report.json"

# ============================================================
# Tokenizer + datasets (shared by both approaches)
# ============================================================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, local_files_only=True)
tokenizer.pad_token_id = SAFE_EOS_ID
tokenizer.eos_token_id = SAFE_EOS_ID

train_ds  = SFTDataset(train_records,    tokenizer, MAX_LENGTH)
train_ds2 = SFTDataset(train_records_s2, tokenizer, MAX_LENGTH)
valid_ds  = SFTDataset(valid_records,    tokenizer, MAX_LENGTH)
collator  = PadCollator(pad_id=SAFE_EOS_ID)

# ============================================================
# Separate TrainingArguments for stage 1 and stage 2
# ============================================================
ta_kwargs_shared = dict(
    output_dir=str(run_output_dir),
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    logging_steps=20,
    save_strategy="epoch",
    save_total_limit=1,
    report_to="none",
    seed=SEED,
    dataloader_num_workers=2,
    remove_unused_columns=False,
)

ta_kwargs_s1 = dict(
    **ta_kwargs_shared,
    per_device_train_batch_size=STAGE1_PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=STAGE1_PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=STAGE1_GRAD_ACCUM,
    warmup_ratio=STAGE1_WARMUP_RATIO,
    weight_decay=STAGE1_WEIGHT_DECAY,
)

ta_kwargs_s2 = dict(
    **ta_kwargs_shared,
    per_device_train_batch_size=STAGE2_PER_DEVICE_BATCH_SIZE,
    per_device_eval_batch_size=STAGE2_PER_DEVICE_BATCH_SIZE,
    gradient_accumulation_steps=STAGE2_GRAD_ACCUM,
    warmup_ratio=STAGE2_WARMUP_RATIO,
    weight_decay=STAGE2_WEIGHT_DECAY,
)

# ============================================================
# Approach A: continued — same adapter, stage2 has higher rank
# via re-init: train s1, merge, attach bigger adapter, train s2
# but keeping it as one logical "continued" adapter trajectory
# ============================================================
if not FREEZE_STAGE_1_ADAPTER :

    lora_config_s1 = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,                   # lower rank for simple arithmetic
        lora_alpha=32,
        # lora_dropout=0.00,
        bias="none",
        target_modules=["c_attn", "c_proj", "c_fc", "mlp.c_proj"],
        fan_in_fan_out=True,
    )

    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    model = get_peft_model(model, lora_config_s1)
    model.print_trainable_parameters()

    # Stage 1 train
    trainer = Trainer(
        model=model,
        args=TrainingArguments(**ta_kwargs_s1, num_train_epochs=STAGE1_EPOCHS, learning_rate=STAGE1_LR, eval_strategy="no"),
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
    )
    t0 = time.time()
    seed_everything(SEED)
    trainer.train()
    print(f"Stage 1 train time: {(time.time()-t0)/60:.2f} min")
    del trainer
    torch.cuda.empty_cache()

    # Stage 2: keep training the same adapter, just update LR
    # (no merge, no new adapter — same parameters continue)
    trainer2 = Trainer(
        model=model,
        args=TrainingArguments(**ta_kwargs_s2, num_train_epochs=STAGE2_EPOCHS, learning_rate=STAGE2_LR, eval_strategy="epoch"),
        train_dataset=train_ds2,
        eval_dataset=valid_ds,
        data_collator=collator,
    )
    t0 = time.time()
    trainer2.train()
    print(f"Stage 2 train time: {(time.time()-t0)/60:.2f} min")

    # Merge final adapter into base
    model = model.merge_and_unload()
    model.save_pretrained(run_output_dir)
    tokenizer.save_pretrained(run_output_dir)
    del trainer2, model
    torch.cuda.empty_cache()

# ============================================================
# Approach B: freeze — merge stage1, fresh higher-rank adapter
# for stage2 so it builds on locked arithmetic knowledge
# ============================================================
else:

    lora_config_s1 = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=16,                   # lower rank for simple arithmetic
        lora_alpha=32,
        lora_dropout=0.00,
        bias="none",
        target_modules=["c_attn", "c_proj", "c_fc", "mlp.c_proj"],
        fan_in_fan_out=True,
    )

    lora_config_s2 = LoraConfig(
        task_type=TaskType.CAUSAL_LM,
        r=128,                   # higher rank for complex reasoning
        lora_alpha=256,
        lora_dropout=0.00,
        bias="none",
        target_modules=["c_attn", "c_proj", "c_fc", "mlp.c_proj"],
        fan_in_fan_out=True,
    )

    # Stage 1 train
    model = AutoModelForCausalLM.from_pretrained(MODEL_NAME, local_files_only=True)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.config.use_cache = False
    model.gradient_checkpointing_enable()
    seed_everything(SEED)
    model = get_peft_model(model, lora_config_s1)
    model.print_trainable_parameters()

    trainer = Trainer(
        model=model,
        args=TrainingArguments(**ta_kwargs_s1, num_train_epochs=STAGE1_EPOCHS, learning_rate=STAGE1_LR, eval_strategy="no"),
        train_dataset=train_ds,
        eval_dataset=valid_ds,
        data_collator=collator,
    )
    t0 = time.time()
    seed_everything(SEED)
    trainer.train()
    print(f"Stage 1 train time: {(time.time()-t0)/60:.2f} min")
    del trainer

    # Merge stage1 permanently into weights — now frozen
    model = model.merge_and_unload()
    model.save_pretrained(stage1_merged_dir)
    tokenizer.save_pretrained(stage1_merged_dir)
    del model
    torch.cuda.empty_cache()

    # Stage 2: fresh adapter on top of merged stage1 base
    model2 = AutoModelForCausalLM.from_pretrained(stage1_merged_dir, local_files_only=True)
    model2.config.pad_token_id = SAFE_EOS_ID
    model2.config.eos_token_id = SAFE_EOS_ID
    model2.config.use_cache = False
    model2.gradient_checkpointing_enable()
    model2 = get_peft_model(model2, lora_config_s2)
    model2.print_trainable_parameters()

    trainer2 = Trainer(
        model=model2,
        args=TrainingArguments(**ta_kwargs_s2, num_train_epochs=STAGE2_EPOCHS, learning_rate=STAGE2_LR, eval_strategy="epoch"),
        train_dataset=train_ds2,
        eval_dataset=valid_ds,
        data_collator=collator,
    )
    t0 = time.time()
    seed_everything(SEED)
    trainer2.train()
    print(f"Stage 2 train time: {(time.time()-t0)/60:.2f} min")

    # Merge stage2 into the already-stage1-merged base → final model
    model2 = model2.merge_and_unload()
    model2.save_pretrained(run_output_dir)
    tokenizer.save_pretrained(run_output_dir)
    del trainer2, model2
    torch.cuda.empty_cache()

# Inference + Evaluate

In [ ]:
import json, re, math, collections
from pathlib import Path

import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
MODEL_PATH  = Path("/kaggle/working/current_run")
VALID_FILE  = Path("/kaggle/input/datasets/kimanh2002/dataset-math/valid.json")
OUTPUT_PATH = Path("/kaggle/working/valid_output.json")
REPORT_PATH = Path("/kaggle/working/valid_report.json")

# ── Inference hyperparameters ──────────────────────────────────────────────
NUM_SAMPLES    = 20       
TEMPERATURE    = 0.7
TOP_K          = 40
TOP_P          = 0.9
BATCH_SIZE     = 4

# ── Model / tokenizer constants ────────────────────────────────────────────
PROMPT_TEMPLATE = "Câu hỏi: {q}\nLời giải: "
SAFE_EOS_ID     = 50256
MAX_LENGTH      = 380
MAX_NEW_TOKENS  = 256

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

In [ ]:
def load_model(model_path, device="cuda"):
    dtype = torch.float16 if device == "cuda" else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    tokenizer.pad_token_id = SAFE_EOS_ID
    tokenizer.eos_token_id = SAFE_EOS_ID
    model = AutoModelForCausalLM.from_pretrained(
        model_path, torch_dtype=dtype, local_files_only=True
    ).to(device)
    model.config.pad_token_id = SAFE_EOS_ID
    model.config.eos_token_id = SAFE_EOS_ID
    model.eval()
    return model, tokenizer

model, tokenizer = load_model(MODEL_PATH, device)
print("Model loaded!")

In [ ]:
def remove_outliers_iqr(values):
    if len(values) < 4:
        return values
    sorted_vals = sorted(values)
    n = len(sorted_vals)
    q1 = sorted_vals[n // 4]
    q3 = sorted_vals[(3 * n) // 4]
    iqr = q3 - q1
    if iqr == 0:
        return values
    lo, hi = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    filtered = [v for v in values if lo <= v <= hi]
    return filtered if filtered else values


def inject_answer_into_text(text, number):
    answer_str = str(int(number)) if number == int(number) else str(number)
    for pattern in [
        r"(Câu trả lời là\s*[:：]?\s*)[\d.,\-]+",
        r"(Đáp án là\s*[:：]?\s*)[\d.,\-]+",
        r"(Đáp án\s*[:：]\s*)[\d.,\-]+",
        r"(The answer is\s*[:：]?\s*)[\d.,\-]+",
        r"(####\s*)[\d.,\-]+",
    ]:
        new_text, n_subs = re.subn(pattern, r"\g<1>" + answer_str, text, count=1)
        if n_subs:
            return new_text
    return text.rstrip() + f"\nĐáp án là: {answer_str}"


def _pick_strategy(sampled_texts, raw_answers, raw_nums):
    valid_pairs = [
        (t, a, n)
        for t, a, n in zip(sampled_texts, raw_answers, raw_nums)
        if n is not None
    ]
    if valid_pairs:
        valid_nums   = [n for _, _, n in valid_pairs]
        cleaned_nums = remove_outliers_iqr(valid_nums)
        cleaned_pairs = [(t, a, n) for t, a, n in valid_pairs if n in cleaned_nums]
    else:
        cleaned_pairs = []

    # 1. Majority vote
    if cleaned_pairs:
        rounded  = [round(n, 4) for _, _, n in cleaned_pairs]
        counter  = collections.Counter(rounded)
        top_cnt  = counter.most_common(1)[0][1]
        top_cands = [v for v, c in counter.items() if c == top_cnt]
        if len(top_cands) == 1:
            majority  = top_cands[0]
            best_text = next(t for t, _, n in cleaned_pairs if round(n, 4) == majority)
            return best_text, majority, "vote", [round(n, 4) for _, _, n in cleaned_pairs]

    # 2. Closest to mean
    if cleaned_pairs:
        nums     = [n for _, _, n in cleaned_pairs]
        mean_val = sum(nums) / len(nums)
        closest  = min(cleaned_pairs, key=lambda x: abs(x[2] - mean_val))
        return closest[0], round(closest[2], 4), "closest_to_mean", [round(n, 4) for _, _, n in cleaned_pairs]

    # 3. Mean override
    if cleaned_pairs:
        nums      = [n for _, _, n in cleaned_pairs]
        mean_val  = sum(nums) / len(nums)
        base_text = min(cleaned_pairs, key=lambda x: abs(x[2] - mean_val))[0]
        return (
            inject_answer_into_text(base_text, mean_val),
            round(mean_val, 4),
            "mean_override",
            [round(n, 4) for _, _, n in cleaned_pairs],
        )

    return sampled_texts[0], None, "no_valid_answer", []


def generate_self_consistency(model, tokenizer, records, device,
                               num_samples, temperature, top_k, top_p,
                               batch_size, max_new_tokens=MAX_NEW_TOKENS):
    vocab_n = model.transformer.wte.num_embeddings
    tokenizer.padding_side = "left"
    outputs = []

    with torch.inference_mode():
        for batch_start in tqdm(range(0, len(records), batch_size),
                                desc="Self-consistency"):
            batch   = records[batch_start : batch_start + batch_size]
            prompts = [PROMPT_TEMPLATE.format(q=r["query_vi"].strip()) for r in batch]

            enc = tokenizer(
                prompts, return_tensors="pt", truncation=True,
                max_length=MAX_LENGTH, padding=True,
            ).to(device)

            input_ids  = enc["input_ids"].clamp(max=vocab_n - 1)
            prompt_len = input_ids.shape[1]

            gen = model.generate(
                input_ids=input_ids,
                attention_mask=enc.get("attention_mask"),
                max_new_tokens=max_new_tokens,
                do_sample=True,
                temperature=temperature,
                top_k=top_k,
                top_p=top_p,
                num_return_sequences=num_samples,
                pad_token_id=SAFE_EOS_ID,
                eos_token_id=SAFE_EOS_ID,
            )

            new_tokens = gen[:, prompt_len:]

            for i, rec in enumerate(batch):
                seq_start     = i * num_samples
                sampled_texts = [
                    tokenizer.decode(new_tokens[seq_start + j], skip_special_tokens=True)
                    for j in range(num_samples)
                ]
                raw_answers = [extract_pred({"model_output": t}) for t in sampled_texts]
                raw_nums    = [parse_number(a) for a in raw_answers]

                best_text, voted_answer, strategy, cleaned_answers = _pick_strategy(
                    sampled_texts, raw_answers, raw_nums
                )
                outputs.append({
                    "id":              len(outputs),
                    "query_vi":        rec["query_vi"],
                    "type":            rec.get("type"),
                    "model_output":    best_text,
                    "all_samples":     sampled_texts,
                    "all_answers":     [round(n, 4) if n is not None else None for n in raw_nums],
                    "cleaned_answers": cleaned_answers,
                    "voted_answer":    voted_answer,
                    "strategy":        strategy,
                })

    return outputs

In [ ]:
sc_outputs = generate_self_consistency(
    model, tokenizer, valid_records, device,
    num_samples=NUM_SAMPLES,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    batch_size=BATCH_SIZE,
)

with OUTPUT_PATH.open("w", encoding="utf-8") as f:
    json.dump(sc_outputs, f, ensure_ascii=False, indent=2)
print(f"Saved {len(sc_outputs)} outputs → {OUTPUT_PATH}")

result = save_evaluation_report(OUTPUT_PATH, valid_records, REPORT_PATH)
print(f"\nFinal score: {result['summary']['raw_score']}/{result['summary']['max_raw_score']} "
      f"({result['summary']['score_pct']*100:.2f}%)")